### STEP 0: INSTALL DEPENDENCIES

This includes `langchain` for orchestrating LLM workflows, `transformers` for pre-trained models, `sentence-transformers` for embeddings, `faiss-cpu` for efficient similarity search, and `pypdf` for PDF loading.

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface transformers sentence-transformers faiss-cpu pypdf

### STEP 1: IMPORT LIBRARIES

Next, we import all the required modules from the installed libraries. This includes `re` for regular expressions, `torch` for PyTorch operations, `google.colab.files` for file uploads, and various components from `langchain` and `transformers`.

In [ ]:
import re
import torch
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

### STEP 2: DEFINE TEXT CLEANING FUNCTION

The function
 `fix_pdf_text` addresses common problems like spaces between characters (e.g., 'C y b e r' to 'Cyber') and ensures cleaner text for processing.

In [ ]:
def fix_pdf_text(text):
    """
    Fix broken PDF text like 'C y b e r' -> 'Cyber'
    """
    text = re.sub(r'(?<= )([^ ]) (?=[^ ])', r'\1', text)
    text = re.sub(r'^([^ ]) (?=[^ ])', r'\1', text)
    text = re.sub(r'\n([^ ]) ', r'\n\1', text)
    return re.sub(r'\s+', ' ', text).strip()

### STEP 3: UPLOAD PDF FILE

 prompts one to select a PDF from your local machine. The uploaded file's path is then stored for further processing.

In [ ]:
print("Upload your PDF file:")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded.")

pdf_path = list(uploaded.keys())[0]

Upload your PDF file:


Saving final_thesis.pdf to final_thesis (1).pdf


### STEP 4: LOAD AND CLEAN PDF

Using `PyPDFLoader`, we load the content of the uploaded PDF. Each page's content is then passed through our `fix_pdf_text` function to clean up any formatting inconsistencies, making the text more suitable for analysis.

In [ ]:
loader = PyPDFLoader(pdf_path)
pages = loader.load()

for page in pages:
    page.page_content = fix_pdf_text(page.page_content)

print(f"Loaded {len(pages)} pages")

Loaded 7 pages


### STEP 5: SPLIT DOCUMENT INTO CHUNKS
 `RecursiveCharacterTextSplitter` is used here to split the PDF content, ensuring chunks are of a specific size with some overlap to maintain context.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = splitter.split_documents(pages)
print(f"Created {len(chunks)} chunks")

Created 7 chunks


### STEP 6: CREATE EMBEDDINGS AND FAISS VECTOR STORE

To enable semantic search, we convert our text chunks into numerical vector representations (embeddings) using a pre-trained `HuggingFaceEmbeddings` model. These embeddings are then stored in a FAISS vector store, which allows for fast similarity searches. A retriever is also configured to fetch the top 5 most relevant chunks.

In [ ]:
print("Creating vector store...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embeddings)

# Improve retrieval quality
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Vector store ready")

Creating vector store...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store ready


### STEP 7: LOAD LARGE LANGUAGE MODEL (FLAN-T5)

 load the `google/flan-t5-large` model, a powerful sequence-to-sequence language model, along with its tokenizer. The model is moved to a GPU if available, otherwise it runs on the CPU.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "google/flan-t5-large"
print(f"Loading model on {device}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

Loading model on cpu...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


### STEP 8: DEFINE RAG (RETRIEVAL-AUGMENTED GENERATION) FUNCTION

The `query_rag` function implements the RAG pattern. It takes a user question, retrieves relevant document chunks using the FAISS retriever, constructs a prompt with the retrieved context, and then uses the FLAN-T5 model to generate an answer *based only* on that context. This helps in grounding the LLM's response in factual information from the PDF.

In [ ]:
def query_rag(question):
    docs = retriever.invoke(question)

    # Format context with clear delimiters for each document
    formatted_context = []
    for i, doc in enumerate(docs):
        formatted_context.append(f"--- Document {i+1} ---\n{doc.page_content}")
    context = "\n\n".join(formatted_context)

    # Show retrieved chunks (for transparency)
    print("\n--- Retrieved Chunks ---")
    for i, doc in enumerate(docs):
        print(f"\nChunk {i+1}:\n{doc.page_content[:200]}...")

    # Refined prompt for more direct instruction
    prompt = f"""
    Use the following pieces of context to answer the user's question.
    If you cannot find the answer from the provided context, please state "I don't know" and do not try to make up an answer.
    Be concise and use bullet points where appropriate.

    Context:
    {context}

    Question:
    {question}

    Your Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


### STEP 9: DEFINE NON-RAG FUNCTION

This `query_without_rag` function provides a baseline by answering the question directly using the FLAN-T5 model without any external document retrieval. This allows us to compare the quality of answers with and without the RAG approach.

In [ ]:
def query_without_rag(question):
    prompt = f"""
Answer the following question:

{question}
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

### STEP 10: RUN COMPARISON

Finally, we define a `user_query` and execute both the `query_without_rag` and `query_rag` functions. This section will print the answers generated by each method, allowing for a direct comparison of their effectiveness in summarizing information from the uploaded PDF.

In [ ]:
user_query = "Summarize the key information, skills, and experience in this document."

print("\n==============================")
print("🔹 WITHOUT RAG")
print("==============================")
print(query_without_rag(user_query))

print("\n==============================")
print("🔹 WITH RAG")
print("==============================")
print(query_rag(user_query))



🔹 WITHOUT RAG
The following is a list of the skills and experience required to be a veterinary surgeon.

🔹 WITH RAG

--- Retrieved Chunks ---

Chunk 1:
CHAPTER FOUR: DATA ANALYSIS 4.1 Introduction Presents findings. 4.2 Findings Institutional capacity exists but effectiveness limited....

Chunk 2:
CHAPTER THREE: RESEARCH METHODOLOGY 3.1 Introduction Explains methods used. 3.2 Research Philosophy Pragmatism. 3.3 Research Design Mixed methods. 3.4 Study Area Somalia, Mali, Sudan, DRC. 3.5 Data So...

Chunk 3:
CHAPTER FIVE: CONCLUSIONS AND RECOMMENDATIONS 5.1 Summary Summarizes findings. 5.2 Conclusions Institutional capacity alone insufficient. 5.3 Recommendations Improve funding and coordination....

Chunk 4:
CHAPTER ONE: INTRODUCTION 1.1 Introduction This chapter introduces the study and outlines key components. 1.2 Background of the Study Conflict remains amajor challenge in Africa... 1.3 Statement of th...

Chunk 5:
Focus on Somalia, Mali, Sudan, DRC. 1.10 Assumptions Data is reliab